# Bulgarian FastSpeech2 phonemes — A100 training

This notebook starts **after punctuation-aware local validation has passed**. The source manifest must contain real punctuation; stripped transcripts are rejected. Select Runtime → Change runtime type → **A100 GPU**. Upload the three files produced by `tools/package_for_colab.py` to `MyDrive/fs2_bg_phone/`. Start ABI-v2 training from step 0; old grapheme and punctuationless-phoneme checkpoints are intentionally rejected.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
gpu_name = torch.cuda.get_device_name(0)
print(torch.__version__, gpu_name)
assert 'A100' in gpu_name, f'Select an A100 runtime, got {gpu_name}'
assert torch.cuda.get_device_capability(0)[0] >= 8, 'GPU lacks native BF16 support'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/YOUR_USER/FastSpeech2.git'  # your fork
BRANCH = 'phoneme-mfa'  # commit and push this branch before opening Colab
DRIVE_DIR = '/content/drive/MyDrive/fs2_bg_phone'

In [ ]:
import os, shutil, subprocess
os.chdir('/content')
shutil.rmtree('FastSpeech2', ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, 'FastSpeech2'], check=True)
os.chdir('/content/FastSpeech2')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
commit_file = f'{DRIVE_DIR}/repo_commit_prosody_v2.txt'
if os.path.isfile(commit_file):
    expected = open(commit_file).read().strip()
    if commit != expected:
        subprocess.run(['git', 'checkout', '--detach', expected], check=True)
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    assert commit == expected, f'Resume must use {expected}, got {commit}'
else:
    open(commit_file, 'w').write(commit + '\n')
print('pinned commit:', commit)

## Install Python dependencies
MFA is deliberately not installed on the A100 runtime. Alignment was completed locally; this avoids a conda runtime restart and leaves the GPU session for preprocessing/training.

In [ ]:
import os, subprocess, sys, zipfile
packages = ['librosa', 'pyworld', 'scikit-learn', 'matplotlib', 'tensorboard', 'soundfile', 'PyYAML', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
with zipfile.ZipFile('hifigan/generator_universal.pth.tar.zip') as zf:
    zf.extractall('hifigan')
assert os.path.isfile('hifigan/generator_universal.pth.tar'), 'vocoder missing'
print('dependencies and vocoder OK')

## Restore the validated local MFA snapshot

In [ ]:
import os, shutil
free_gib = shutil.disk_usage('/content').free / 1024**3
print('local disk free:', round(free_gib, 1), 'GiB')
assert free_gib >= 45, 'Need at least 45 GiB free local disk'
for name in ['raw_data_Bulgarian.zip', 'TextGrid_Bulgarian.zip', 'phoneme_assets.zip']:
    path = f'{DRIVE_DIR}/{name}'
    assert os.path.isfile(path), f'Missing {path}'
local_snapshot = '/content/mfa_phone_snapshot'
shutil.rmtree(local_snapshot, ignore_errors=True)
os.makedirs(local_snapshot)
for name in ['raw_data_Bulgarian.zip', 'TextGrid_Bulgarian.zip', 'phoneme_assets.zip']:
    print('copying', name)
    shutil.copy2(f'{DRIVE_DIR}/{name}', f'{local_snapshot}/{name}')
shutil.rmtree('raw_data/Bulgarian', ignore_errors=True)
shutil.rmtree('preprocessed_data/Bulgarian', ignore_errors=True)
shutil.unpack_archive(f'{local_snapshot}/raw_data_Bulgarian.zip', 'raw_data')
os.makedirs('preprocessed_data/Bulgarian', exist_ok=True)
shutil.unpack_archive(f'{local_snapshot}/TextGrid_Bulgarian.zip', 'preprocessed_data/Bulgarian')
shutil.unpack_archive(f'{local_snapshot}/phoneme_assets.zip', '.')
shutil.rmtree(local_snapshot)
print('snapshot restored')

In [ ]:
import subprocess
subprocess.run(['python', 'tools/validate_mfa_pipeline.py', '--stage', 'alignment'], check=True)

## Preprocess on A100
This rebuilds mel, pitch, energy and duration from an empty feature state. TextGrids are preserved. Depending on Drive/extraction and CPU speed, feature extraction can still take a while even though STFT uses CUDA.

In [ ]:
import subprocess
subprocess.run(['python', 'preprocess.py', 'config/Bulgarian/preprocess.yaml'], check=True)
subprocess.run(['python', 'tools/validate_mfa_pipeline.py', '--stage', 'preprocessed'], check=True)
subprocess.run(['wc', '-l', 'preprocessed_data/Bulgarian/train.txt', 'preprocessed_data/Bulgarian/val.txt'], check=True)

## Back up preprocessed features
Run once after validation. On future A100 sessions restore this archive and skip raw/TextGrid extraction plus preprocessing.

In [ ]:
import os, shutil, zipfile
local_base = '/content/preprocessed_Bulgarian_prosody_v2'
local_zip = local_base + '.zip'
drive_zip = f'{DRIVE_DIR}/preprocessed_Bulgarian_prosody_v2.zip'
drive_tmp = drive_zip + '.tmp'
if os.path.exists(local_zip): os.remove(local_zip)
shutil.make_archive(local_base, 'zip', 'preprocessed_data', 'Bulgarian')
with zipfile.ZipFile(local_zip) as zf:
    assert zf.testzip() is None, 'local feature archive failed CRC validation'
if os.path.exists(drive_tmp): os.remove(drive_tmp)
shutil.copy2(local_zip, drive_tmp)
assert os.path.getsize(local_zip) == os.path.getsize(drive_tmp)
os.replace(drive_tmp, drive_zip)
print('created and verified', drive_zip, os.path.getsize(drive_zip) / 1e9, 'GB')

## A100 training configuration
BF16 is enabled. Batch 64 is the conservative starting point; try 80 or 96 only after a few successful validation/checkpoint cycles. Use the identical values whenever resuming because checkpoints validate their pipeline/config hashes.

In [ ]:
import yaml
train_path = 'config/Bulgarian/train.yaml'
with open(train_path) as f: train_cfg = yaml.safe_load(f)
train_cfg['optimizer'].update({'batch_size': 64, 'num_workers': 2, 'amp': True, 'amp_dtype': 'bfloat16'})
train_cfg['step'].update({'total_step': 60000, 'save_step': 2000, 'val_step': 1000, 'synth_step': 1000})
with open(train_path, 'w') as f: yaml.safe_dump(train_cfg, f, sort_keys=False)
print(train_cfg['optimizer'])
print(train_cfg['step'])

## Mandatory training smoke test
Run one real BF16 forward/backward pass at the configured batch size before spending hours on training. If this OOMs, set batch size to 32 in the configuration cell and rerun both cells.

In [ ]:
import subprocess
subprocess.run(['python', 'tools/smoke_test_training.py', '--batch-size', str(train_cfg['optimizer']['batch_size'])], check=True)

In [ ]:
import os
os.makedirs(f'{DRIVE_DIR}/output_prosody_v2', exist_ok=True)
!rm -rf output
!ln -s "{DRIVE_DIR}/output_prosody_v2" output
!ls -la output

## Start from zero
Run this only for the first ABI-v2 prosody run. Do not point it at a grapheme or old punctuationless-phoneme checkpoint.

In [ ]:
import glob
existing = glob.glob('output/ckpt/Bulgarian/*.pth.tar')
assert not existing, f'Existing checkpoints found; use Resume instead: {existing[-3:]}'
import os, subprocess
train_command = ['python', 'train.py', '-p', 'config/Bulgarian/preprocess.yaml', '-m', 'config/Bulgarian/model.yaml', '-t', 'config/Bulgarian/train.yaml']
subprocess.run(train_command, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})

## Resume after disconnect
On a fresh runtime repeat GPU check, Drive mount, clone, dependency install and the exact configuration cell. Restore `preprocessed_Bulgarian_prosody_v2.zip`, recreate the output symlink, then run this cell.

In [ ]:
import os, re, subprocess
ckpt_dir = 'output/ckpt/Bulgarian'
import torch
steps = sorted((int(re.match(r'(\d+)\.pth\.tar$', x).group(1)) for x in os.listdir(ckpt_dir) if re.match(r'(\d+)\.pth\.tar$', x)), reverse=True)
RESTORE_STEP = None
for candidate in steps:
    try:
        state = torch.load(f'{ckpt_dir}/{candidate}.pth.tar', map_location='cpu', weights_only=False)
        assert {'model', 'optimizer', 'pipeline'} <= set(state)
        RESTORE_STEP = candidate
        break
    except Exception as exc:
        print('skipping invalid checkpoint', candidate, repr(exc))
assert RESTORE_STEP is not None, 'No valid phoneme checkpoint found'
print('resuming', RESTORE_STEP)
train_command = ['python', 'train.py', '--restore_step', str(RESTORE_STEP), '-p', 'config/Bulgarian/preprocess.yaml', '-m', 'config/Bulgarian/model.yaml', '-t', 'config/Bulgarian/train.yaml']
subprocess.run(train_command, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})

In [ ]:
%load_ext tensorboard
%tensorboard --logdir output/log/Bulgarian

## Quick synthesis from the latest checkpoint
The sample sentence uses words already present in the packaged runtime lexicon, so MFA is not needed. Arbitrary out-of-vocabulary words still require the Bulgarian MFA G2P model.

In [ ]:
import glob, os, re, subprocess, torch
from IPython.display import Audio, display
ckpts = sorted((int(re.search(r'/(\d+)\.pth\.tar$', p).group(1)), p) for p in glob.glob('output/ckpt/Bulgarian/*.pth.tar'))
assert ckpts, 'No checkpoint found yet'
SYNTH_STEP = ckpts[-1][0]
text = 'Той каза, че това е истина.'
command = ['python', 'synthesize.py', '--restore_step', str(SYNTH_STEP), '--mode', 'single', '--text', text, '-p', 'config/Bulgarian/preprocess.yaml', '-m', 'config/Bulgarian/model.yaml', '-t', 'config/Bulgarian/train.yaml']
subprocess.run(command, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})
wav = max(glob.glob('output/result/Bulgarian/*.wav'), key=os.path.getmtime)
print(wav)
display(Audio(wav))

## Fast resume: restore preprocessed features
Use this instead of raw/TextGrid extraction and preprocessing when `preprocessed_Bulgarian_prosody_v2.zip` already exists. The small assets archive is still restored for the runtime lexicon and prosody metadata.

In [ ]:
import os, shutil, subprocess
drive_feature_zip = f'{DRIVE_DIR}/preprocessed_Bulgarian_prosody_v2.zip'
local_feature_zip = '/content/preprocessed_Bulgarian_prosody_v2.zip'
assert os.path.isfile(drive_feature_zip), f'Missing {drive_feature_zip}'
shutil.copy2(drive_feature_zip, local_feature_zip)
shutil.rmtree('preprocessed_data/Bulgarian', ignore_errors=True)
shutil.unpack_archive(local_feature_zip, 'preprocessed_data')
asset_zip = f'{DRIVE_DIR}/phoneme_assets.zip'
assert os.path.isfile(asset_zip), f'Missing {asset_zip}'
shutil.unpack_archive(asset_zip, '.')
os.remove(local_feature_zip)
subprocess.run(['python', 'tools/validate_mfa_pipeline.py', '--stage', 'preprocessed'], check=True)